In [ ]:
!pip install -q sentence-transformers

In [ ]:
!pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.4 MB/s eta 0:00:00


In [ ]:
import os
import pickle
import faiss
import numpy as np
import pandas as pd

from tqdm import tqdm

from sentence_transformers import CrossEncoder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATA_PATH = "/content/drive/MyDrive/FIFI_Research/data"

val_df = pd.read_csv(
    os.path.join(DATA_PATH, "val.tsv"),
    sep="\t"
)

print(val_df.shape)

(18000, 4)


In [ ]:
MODEL_PATH="/content/drive/MyDrive/FIFI_Research/models"

with open(
    os.path.join(MODEL_PATH,"candidate_titles.pkl"),
    "rb"
) as f:
    candidate_titles = pickle.load(f)

candidate_embeddings=np.load(
    os.path.join(MODEL_PATH,"candidate_embeddings.npy")
)

query_embeddings=np.load(
    os.path.join(MODEL_PATH,"query_embeddings.npy")
)

faiss_scores=np.load(
    os.path.join(MODEL_PATH,"faiss_scores.npy")
)

faiss_indices=np.load(
    os.path.join(MODEL_PATH,"faiss_indices.npy")
)

index=faiss.read_index(
    os.path.join(MODEL_PATH,"faiss.index")
)

print("Everything Loaded Successfully!")

Everything Loaded Successfully!


In [ ]:
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
pairs=[]

mapping=[]

for query_idx in range(len(val_df)):

    generated=val_df.iloc[query_idx]["generated_title"]

    retrieved=faiss_indices[query_idx]

    for candidate_idx in retrieved:

        pairs.append([

            generated,

            candidate_titles[candidate_idx]

        ])

        mapping.append(query_idx)

print("Total pairs:",len(pairs))

Total pairs: 360000


In [ ]:
cross_scores = cross_encoder.predict(
    pairs,
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/11250 [00:00<?, ?it/s]

In [ ]:
reranked_titles=[]

reranked_scores=[]

pointer=0

TOP_K=10

for query_idx in tqdm(range(len(val_df))):

    candidates=[]

    for i in range(TOP_K):

        candidate_index=faiss_indices[query_idx][i]

        candidates.append({

            "title":candidate_titles[candidate_index],

            "score":cross_scores[pointer]

        })

        pointer+=1

    candidates=sorted(

        candidates,

        key=lambda x:x["score"],

        reverse=True

    )

    reranked_titles.append(

        [c["title"] for c in candidates]

    )

    reranked_scores.append(

        [c["score"] for c in candidates]

    )

100%|██████████| 18000/18000 [00:00<00:00, 45111.01it/s]


In [ ]:
sample=0

print("Generated\n")

print(val_df.iloc[sample]["generated_title"])

print("\nGround Truth\n")

print(val_df.iloc[sample]["original_title"])

print("\nTop 10 After Reranking\n")

for i,title in enumerate(reranked_titles[sample]):

    print(i+1,title)

Generated

Fully Convolutional Joint Detection and Regression for Preterm Infant Limb-Pose Estimation from Depth Images

Ground Truth

Preterm infants' limb-pose estimation from depth images using  convolutional neural networks

Top 10 After Reranking

1 Preterm infants' limb-pose estimation from depth images using  convolutional neural networks
2 Back to the Future: Joint Aware Temporal Deep Learning 3D Human Pose  Estimation
3 Peeking into occluded joints: A novel framework for crowd pose  estimation
4 Real-time Deep Pose Estimation with Geodesic Loss for Image-to-Template  Rigid Registration
5 Capture Dense: Markerless Motion Capture Meets Dense Pose Estimation
6 Collaborative Descriptors: Convolutional Maps for Preprocessing
7 Iterative Multi-domain Regularized Deep Learning for Anatomical  Structure Detection and Segmentation from Ultrasound Images
8 A Deep Framework for Bone Age Assessment based on Finger Joint  Localization
9 Learning Transferable Kinematic Dictionary for 3D Hum

In [ ]:
def reciprocal_rank(predictions,ground_truth):

    for rank,title in enumerate(predictions,start=1):

        if title==ground_truth:

            return 1/rank

    return 0

In [ ]:
rr=[]

for idx in range(len(val_df)):

    rr.append(

        reciprocal_rank(

            reranked_titles[idx],

            val_df.iloc[idx]["original_title"]

        )

    )

mrr=np.mean(rr)

print("CrossEncoder MRR@10 :",round(mrr,4))

CrossEncoder MRR@10 : 0.4831


In [ ]:
style_results=[]

for style in [

    "technical",

    "accessible",

    "catchy"

]:

    subset=val_df[val_df["category"]==style]

    scores=[]

    for idx in subset.index:

        scores.append(

            reciprocal_rank(

                reranked_titles[idx],

                val_df.loc[idx,"original_title"]

            )

        )

    style_results.append({

        "Style":style,

        "MRR@10":np.mean(scores)

    })

style_results=pd.DataFrame(style_results)

style_results

,Style,MRR@10
0,technical,0.542371
1,accessible,0.440262
2,catchy,0.466724


In [ ]:
comparison=pd.DataFrame({

    "Model":[

        "BGE + FAISS",

        "BGE + FAISS + CrossEncoder"

    ],

    "MRR@10":[

        0.8222,

        mrr

    ]

})

comparison

,Model,MRR@10
0,BGE + FAISS,0.822200
1,BGE + FAISS + CrossEncoder,0.483119


In [ ]:
submission=[]

for idx in range(len(val_df)):

    query_id=val_df.iloc[idx]["id"]

    for rank in range(10):

        submission.append({

            "id":query_id,

            "rank":rank+1,

            "score":float(reranked_scores[idx][rank]),

            "original_title":reranked_titles[idx][rank]

        })

submission_df=pd.DataFrame(submission)

submission_df.to_csv(

"/content/drive/MyDrive/FIFI_Research/submissions/FutureMinds_task1_run.tsv",

sep="\t",

index=False

)

submission_df.head()

,id,rank,score,original_title
0,0,1,4.517137,Preterm infants' limb-pose estimation from dep...
1,0,2,-3.975867,Back to the Future: Joint Aware Temporal Deep ...
2,0,3,-5.231252,Peeking into occluded joints: A novel framewor...
3,0,4,-6.015102,Real-time Deep Pose Estimation with Geodesic L...
4,0,5,-7.596562,Capture Dense: Markerless Motion Capture Meets...


In [ ]:
errors=[]

for idx in range(len(val_df)):

    gt=val_df.iloc[idx]["original_title"]

    if gt not in reranked_titles[idx]:

        errors.append({

            "generated_title":

            val_df.iloc[idx]["generated_title"],

            "ground_truth":gt,

            "top_prediction":

            reranked_titles[idx][0]

        })

errors_df=pd.DataFrame(errors)

errors_df.head(20)

,generated_title,ground_truth,top_prediction
0,A Compact AI System That Predicts What Happens...,"VP-GO: a ""light"" action-conditioned visual pre...",SoftGym: Benchmarking Deep Reinforcement Learn...
1,Training AI Models to Work on New Data by Sepa...,DecAug: Out-of-Distribution Generalization via...,Unadversarial Examples: Designing Objects for ...
2,A More Accurate Way to Discover Network Struct...,An Empirical-Bayes Score for Discrete Bayesian...,Recognizing Predictive Substructures with Subg...
3,Teaching Computers to Automatically Identify a...,DisCont: Self-Supervised Visual Attribute Dise...,Recognizing Objects In-the-wild: Where Do We S...
4,Saving Your Questions for When They Matter: Sm...,Confidence-Budget Matching for Sequential Budg...,HybridSVD: When Collaborative Information is N...
5,A New Way to Learn Better Image Keypoints and ...,HDD-Net: Hybrid Detector Descriptor with Mutua...,Image-embodied Knowledge Representation Learning
6,Using Deep Learning to Help Doctors Screen Med...,An Analysis of a BERT Deep Learning Strategy o...,Comparison of Deep Learning Approaches for Mul...
7,Teaching Computers to Understand How People In...,HOTR: End-to-End Human-Object Interaction Dete...,Training and Testing Object Detectors with Vir...
8,Improving Digital Image Resolution Using More ...,NeuroTreeNet: A New Method to Explore Horizont...,Unsupervised Image Super-Resolution using Cycl...
9,A New Reasoning Method for Helping AI Deal wit...,A Novel Fuzzy Approximate Reasoning Method Bas...,Truthful AI: Developing and governing AI that ...
